# 08 - Swaps Verification

This notebook demonstrates the Multi-Curve Integration and AAD framework for Interest Rate Swaps.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.derivatives.swap import InterestRateSwap, swap_pv
from src.curve.nelson_siegel import NelsonSiegelCurve
from src.derivatives.aad import Dual

## 1. Multi-Curve Pricing

In [2]:
# Discount and Forward curves
discount_curve = NelsonSiegelCurve(0.03, 0.0, 0.0, 1.0)
forward_curve = NelsonSiegelCurve(0.04, 0.0, 0.0, 1.0)

swap = InterestRateSwap(notional=1e6, fixed_rate=0.035, tenor=5.0, freq=2)

pv = swap_pv(swap, discount_curve, forward_curve, position="receiver")
print(f"Swap PV (Multi-Curve): {float(pv):.2f}")


Swap PV (Multi-Curve): -24897.34


## 2. AAD Exact Greeks

In [3]:
# Using Dual numbers to get exact sensitivities (Rho) to parallel curve shifts
bump_d = Dual(0.0)
bump_f = Dual(0.0)

def bumped_discount(t):
    # AAD-enabled discount curve wrapper
    return 0.03 + bump_d

def bumped_forward(t):
    return 0.04 + bump_f

pv_dual = swap_pv(swap, bumped_discount, bumped_forward, position="receiver")
pv_dual.backward()

print(f"PV: {float(pv_dual):.2f}")
print(f"DV01 w.r.t Discount Curve: {bump_d.adjoint:.2f}")
print(f"DV01 w.r.t Forward Curve: {bump_f.adjoint:.2f}")


PV: -24897.34
DV01 w.r.t Discount Curve: 66927.75
DV01 w.r.t Forward Curve: -4701425.98
